# Agent function calls

The kernel handles `getForecast` calls sent by the renderer and replies with `agentFunctionResponse` using the same `functionCallId`. Click **Seattle** or **Lisbon** to update the forecast.

In [ ]:
const adapterUrl = globalThis.document
  ? new URL('../files/a2ui/kernel.js', document.baseURI)
  : new URL('../../../../files/a2ui/kernel.js', location.href);
const { A2UISurface } = await import(adapterUrl.href);
globalThis.surface = new A2UISurface();
surface.send([
  {
    "version": "1.0",
    "createSurface": {
      "surfaceId": "agent-function-call",
      "catalogId": "https://a2ui.org/specification/v1_0/catalogs/basic/catalog.json"
    }
  },
  {
    "version": "1.0",
    "updateDataModel": {
      "surfaceId": "agent-function-call",
      "value": {
        "forecast": "No forecast yet"
      }
    }
  },
  {
    "version": "1.0",
    "updateComponents": {
      "surfaceId": "agent-function-call",
      "components": [
        {
          "id": "root",
          "component": "Column",
          "children": [
            "title",
            "seattle_button",
            "lisbon_button",
            "forecast"
          ],
          "align": "center"
        },
        {
          "id": "title",
          "component": "Text",
          "text": "## Forecast lookup"
        },
        {
          "id": "seattle_button",
          "component": "Button",
          "child": "seattle_label",
          "action": {
            "functionCall": {
              "@call": "getForecast",
              "args": {
                "city": "Seattle"
              }
            }
          }
        },
        {
          "id": "seattle_label",
          "component": "Text",
          "text": "Seattle"
        },
        {
          "id": "lisbon_button",
          "component": "Button",
          "child": "lisbon_label",
          "action": {
            "functionCall": {
              "@call": "getForecast",
              "args": {
                "city": "Lisbon"
              }
            }
          }
        },
        {
          "id": "lisbon_label",
          "component": "Text",
          "text": "Lisbon"
        },
        {
          "id": "forecast",
          "component": "Text",
          "text": {
            "@path": "/forecast"
          },
          "variant": "caption"
        }
      ]
    }
  }
]);

const forecasts = {Seattle: 'Light rain, 12 C', Lisbon: 'Sunny, 21 C'};
globalThis.unsubscribe = surface.onMessage(message => {
  if (!('callAgentFunction' in message)) return;
  const call = message.callAgentFunction;
  if (call.callFunction['@call'] !== 'getForecast') return;
  const city = call.callFunction.args.city;
  const forecast = forecasts[city] ?? 'Unknown city';
  surface.send([
    {version: '1.0', agentFunctionResponse: {functionCallId: call.functionCallId, value: forecast}},
    {version: '1.0', updateDataModel: {surfaceId: 'agent-function-call', path: '/forecast', value: `${city}: ${forecast}`}}
  ]);
});
display(surface);

Remove the forecast handler.

In [ ]:
unsubscribe();
console.log('Forecast handler removed.');